# UCS420 – Cognitive Computing | Assignment 10
## Diabetes Care Information Assistant

# Student Name: Kovid Kumar  
# Roll Number: 1024170353  

## 1. Setup and imports


In [ ]:
!pip -q install nltk scikit-learn pandas matplotlib seaborn

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import wordpunct_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Setup complete.")

## 2. Labelled query corpus


In [ ]:
data = [
    # glucose_monitoring (9)
    ("How can I check my blood sugar at home?", "glucose_monitoring"),
    ("When should I use a glucometer?", "glucose_monitoring"),
    ("What does fasting blood glucose mean?", "glucose_monitoring"),
    ("How do I record my glucose readings?", "glucose_monitoring"),
    ("What is a post meal blood sugar test?", "glucose_monitoring"),
    ("Can I track my sugar levels daily?", "glucose_monitoring"),
    ("What is HbA1c used for?", "glucose_monitoring"),
    ("How should I prepare for a glucose test?", "glucose_monitoring"),
    ("Explain continuous glucose monitoring.", "glucose_monitoring"),

    # symptoms_and_risks (9)
    ("What are common symptoms of diabetes?", "symptoms_and_risks"),
    ("Why do I feel thirsty all the time?", "symptoms_and_risks"),
    ("What are signs of low blood sugar?", "symptoms_and_risks"),
    ("What are signs of high blood sugar?", "symptoms_and_risks"),
    ("Can diabetes cause blurry vision?", "symptoms_and_risks"),
    ("What is diabetic neuropathy?", "symptoms_and_risks"),
    ("Why does diabetes cause frequent urination?", "symptoms_and_risks"),
    ("What are long term complications of diabetes?", "symptoms_and_risks"),
    ("Can diabetes affect the kidneys?", "symptoms_and_risks"),

    # nutrition (8)
    ("What foods are helpful for diabetes management?", "nutrition"),
    ("Can a person with diabetes eat fruit?", "nutrition"),
    ("Suggest healthy breakfast ideas for diabetes.", "nutrition"),
    ("Why is fibre important in meals?", "nutrition"),
    ("Should I avoid sugary drinks?", "nutrition"),
    ("What is a balanced plate for diabetes?", "nutrition"),
    ("Are whole grains a better choice?", "nutrition"),
    ("How can I reduce added sugar in my diet?", "nutrition"),

    # physical_activity (8)
    ("What exercises are suitable for people with diabetes?", "physical_activity"),
    ("Can walking help manage blood glucose?", "physical_activity"),
    ("How often should adults be physically active?", "physical_activity"),
    ("Should I check glucose around exercise?", "physical_activity"),
    ("What are some beginner activities?", "physical_activity"),
    ("Can strength training be useful?", "physical_activity"),
    ("How can I stay active during the day?", "physical_activity"),
    ("Is it important to warm up before exercise?", "physical_activity"),

    # medication_and_care (8)
    ("Why is it important to take prescribed diabetes medicine?", "medication_and_care"),
    ("What should I ask my doctor about insulin?", "medication_and_care"),
    ("Can I stop my medicine when I feel better?", "medication_and_care"),
    ("How should I remember my medicines?", "medication_and_care"),
    ("Why are regular checkups important?", "medication_and_care"),
    ("What should I discuss during a diabetes appointment?", "medication_and_care"),
    ("How can I care for my feet?", "medication_and_care"),
    ("Why should I follow my treatment plan?", "medication_and_care"),

    # general_diabetes (8)
    ("What is diabetes?", "general_diabetes"),
    ("Explain type 1 diabetes.", "general_diabetes"),
    ("What is type 2 diabetes?", "general_diabetes"),
    ("What is insulin and what does it do?", "general_diabetes"),
    ("What is the difference between type 1 and type 2?", "general_diabetes"),
    ("Can diabetes be managed?", "general_diabetes"),
    ("What does insulin resistance mean?", "general_diabetes"),
    ("Can you explain diabetes in simple words?", "general_diabetes"),
]

df = pd.DataFrame(data, columns=["query", "intent"])
print("Number of labelled queries:", len(df))
print("Number of intent categories:", df["intent"].nunique())
display(df.groupby("intent").size().rename("query_count").to_frame())
display(df.head())

## 3. Hierarchical taxonomy

The taxonomy has **4 major categories**, each with at least **2 subcategories**. The six intents used by the classifier are mapped into this hierarchy.

In [ ]:
taxonomy = {
    "Understanding diabetes": [
        "general_diabetes: definition, types, insulin and insulin resistance",
        "symptoms_and_risks: symptoms, low/high glucose signs and complications"
    ],
    "Monitoring and assessment": [
        "glucose_monitoring: glucometer, fasting/post-meal tests, HbA1c and CGM",
        "screening_and_follow_up: routine tests and clinician review"
    ],
    "Lifestyle management": [
        "nutrition: balanced meals, fibre, fruits and added sugar",
        "physical_activity: walking, exercise planning and activity safety"
    ],
    "Treatment and ongoing care": [
        "medication_and_care: prescribed medicines, insulin discussions and foot care",
        "self_management_support: reminders, appointments and care planning"
    ]
}
for major, subs in taxonomy.items():
    print(major)
    for sub in subs:
        print("  └──", sub)

## 4. Domain concepts and relationships

The following small domain model contains more than 8 concepts and 10 relationships. It is a simplified educational knowledge representation.

In [ ]:
domain_concepts = {
    "Diabetes": "A group of conditions involving elevated blood glucose.",
    "Blood glucose": "The amount of glucose in the blood.",
    "Insulin": "A hormone that helps glucose enter cells.",
    "Pancreas": "An organ that produces insulin.",
    "HbA1c": "A blood test reflecting average glucose over roughly 2–3 months.",
    "Glucose meter": "A device used to measure a blood glucose sample.",
    "Nutrition": "Food and meal choices relevant to health.",
    "Physical activity": "Movement and exercise that support general health.",
    "Medication": "Treatment prescribed by a qualified clinician.",
    "Complications": "Health problems that may develop with diabetes over time.",
    "Healthcare professional": "A qualified clinician involved in care."
}

relationships = [
    ("Diabetes", "is associated with", "Blood glucose"),
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "helps regulate", "Blood glucose"),
    ("Glucose meter", "measures", "Blood glucose"),
    ("HbA1c", "provides information about", "Blood glucose"),
    ("Nutrition", "can influence", "Blood glucose"),
    ("Physical activity", "can influence", "Blood glucose"),
    ("Medication", "may be prescribed to manage", "Diabetes"),
    ("Diabetes", "may lead to", "Complications"),
    ("Healthcare professional", "assesses", "Diabetes"),
    ("Healthcare professional", "advises on", "Medication"),
    ("Healthcare professional", "may order", "HbA1c")
]

concept_df = pd.DataFrame(
    [(k, v) for k, v in domain_concepts.items()],
    columns=["Concept", "Definition"]
)
relation_df = pd.DataFrame(relationships, columns=["Source", "Relationship", "Target"])
print("Number of concepts:", len(domain_concepts))
print("Number of relationships:", len(relationships))
display(concept_df)
display(relation_df)

## 5. NLP preprocessing

The preprocessing pipeline performs:
1. Lowercase conversion
2. Tokenization
3. Stopword removal
4. Lemmatization



In [ ]:
stop_words = set(stopwords.words("english")) - {"not", "no", "nor"}
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    tokens = wordpunct_tokenize(text)
    cleaned = []
    for token in tokens:
        if re.fullmatch(r"[a-z]+", token) and token not in stop_words:
            cleaned.append(lemmatizer.lemmatize(token))
    return " ".join(cleaned)

df["processed_query"] = df["query"].apply(preprocess_text)
display(df[["query", "processed_query", "intent"]].head(10))

## 6. TF-IDF feature extraction and intent classifier


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["query"], df["intent"],
    test_size=0.25,
    random_state=42,
    stratify=df["intent"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=preprocess_text,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Training complete.")
print("Training examples:", len(X_train), "| Test examples:", len(X_test))

## 7. Model evaluation

The model is evaluated using accuracy, precision, recall, F1-score and a confusion matrix. 

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"],
    "Score": [accuracy, precision, recall, f1]
})
display(metrics_df.style.format({"Score": "{:.3f}"}))
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(df["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("Predicted intent")
ax.set_ylabel("Actual intent")
ax.set_title("Confusion Matrix")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Diabetes information knowledge base



In [ ]:
knowledge_base = {
    "general_diabetes": (
        "Diabetes is a condition in which blood glucose is higher than normal because "
        "the body does not make enough insulin or cannot use insulin effectively. "
        "Type 1 and type 2 diabetes have different causes and care needs. A healthcare "
        "professional can explain the type and appropriate care plan."
    ),
    "glucose_monitoring": (
        "Blood glucose can be checked with a home glucose meter or, for some people, "
        "a continuous glucose monitor (CGM). HbA1c is a laboratory test that gives an "
        "estimate of average blood glucose over about 2–3 months. Follow the testing "
        "schedule and interpretation advice provided by your healthcare professional."
    ),
    "symptoms_and_risks": (
        "Possible diabetes symptoms include increased thirst, frequent urination, "
        "increased hunger, fatigue, blurred vision and unexplained weight changes. "
        "Low blood glucose can cause shakiness, sweating, hunger, dizziness or confusion; "
        "high glucose may cause thirst and frequent urination. Severe symptoms need urgent care."
    ),
    "nutrition": (
        "A balanced eating pattern may include non-starchy vegetables, high-fibre foods, "
        "whole grains, suitable protein sources and water instead of sugary drinks. "
        "Food needs differ between people; a registered dietitian or clinician can help "
        "create an individualized plan."
    ),
    "physical_activity": (
        "Regular physical activity can support general health and blood glucose management. "
        "Walking, cycling, swimming and strength activities may be options depending on "
        "health and ability. Ask your healthcare professional how to exercise safely, "
        "especially if you use glucose-lowering medication or have complications."
    ),
    "medication_and_care": (
        "Take medicines only as prescribed. Do not start, stop or change a diabetes "
        "medicine or insulin dose based on this assistant. Discuss side effects, missed "
        "doses, monitoring, foot care and routine checkups with your healthcare professional."
    )
}

display(pd.DataFrame(
    [{"Intent": k, "Knowledge base response": v} for k, v in knowledge_base.items()]
))

## 9. Reasoning rules and safety handling

At least five rules are implemented:
- Emergency escalation for potentially urgent symptoms.
- Low-confidence handling.
- Unsupported-query handling.
- Medication-change safety rule.
- General health-information response with a clinician reminder.

The emergency keyword rule is intentionally cautious and is not a substitute for emergency triage.

In [ ]:
EMERGENCY_KEYWORDS = [
    "unconscious", "passed out", "seizure", "convulsion",
    "cannot breathe", "can't breathe", "chest pain",
    "severe confusion", "unresponsive"
]

MEDICATION_CHANGE_KEYWORDS = [
    "stop my medicine", "stop medication", "change my dose",
    "increase my dose", "decrease my dose", "double my dose",
    "skip insulin", "change insulin dose"
]

SUPPORTED_KEYWORDS = [
    "diabetes", "glucose", "blood sugar", "insulin", "hba1c",
    "glucometer", "sugar", "diabetic", "hypoglycemia",
    "hyperglycemia", "food", "exercise", "medicine"
]

CONFIDENCE_THRESHOLD = 0.35

def is_emergency_query(query):
    q = query.lower()
    return any(term in q for term in EMERGENCY_KEYWORDS)

def is_medication_change_query(query):
    q = query.lower()
    return any(term in q for term in MEDICATION_CHANGE_KEYWORDS)

def is_supported_domain_query(query):
    q = query.lower()
    return any(term in q for term in SUPPORTED_KEYWORDS)

def classify_intent(query):
    probabilities = model.predict_proba([query])[0]
    classes = model.named_steps["classifier"].classes_
    best_idx = int(np.argmax(probabilities))
    return classes[best_idx], float(probabilities[best_idx])

def diabetes_assistant(query):
    query = str(query).strip()
    if not query:
        return "Please enter a question about diabetes care."

    # Rule 1: Emergency escalation takes priority over model prediction.
    if is_emergency_query(query):
        return (
            "URGENT SAFETY MESSAGE: This may describe a medical emergency. "
            "Seek emergency medical help now or go to the nearest emergency department. "
            "Do not rely on this chatbot for emergency assessment. In India, call 112 "
            "if emergency assistance is needed."
        )

    # Rule 2: Never recommend changing medication.
    if is_medication_change_query(query):
        return (
            "Medication safety: Do not stop, start or change a diabetes medicine or "
            "insulin dose based on chatbot advice. Contact your prescribing clinician "
            "or pharmacist for personal instructions."
        )

    # Rule 3: Unsupported-domain handling.
    if not is_supported_domain_query(query):
        return (
            "I can currently answer general questions about diabetes, glucose monitoring, "
            "symptoms and risks, nutrition, physical activity, and ongoing care. "
            "Please ask a diabetes-related question."
        )

    intent, confidence = classify_intent(query)

    # Rule 4: Low-confidence handling.
    if confidence < CONFIDENCE_THRESHOLD:
        return (
            "I'm not sufficiently confident about the meaning of that question. "
            "Could you rephrase it or ask a healthcare professional? "
            "I do not want to guess about health information."
        )

    # Rule 5: Retrieve intent-specific information.
    response = knowledge_base.get(intent)
    if response is None:
        return "I do not have reliable information for that topic. Please consult a clinician."

    return (
        f"Detected intent: {intent.replace('_', ' ').title()} "
        f"(model confidence: {confidence:.2f})\n\n"
        f"{response}\n\n"
        "Reminder: This is general educational information, not a diagnosis or personal "
        "treatment plan. Consult your healthcare professional for advice about your situation."
    )

print("Safety rules and assistant function are ready.")

## 10. Test cases required by the assignment

The assistant is tested with:
1. A query from each defined intent.
2. A differently worded query.
3. An unsupported healthcare query.
4. A low-confidence query.
5. An emergency-related query.

In [ ]:
test_queries = [
    "What is type 2 diabetes?",  # general_diabetes
    "How do I check my sugar at home?",  # glucose_monitoring
    "Why am I always thirsty?",  # symptoms_and_risks
    "What should I eat for breakfast?",  # nutrition
    "Is walking useful for blood sugar?",  # physical_activity
    "Should I stop my diabetes medicine if I feel fine?",  # medication safety
    "Tell me about diabetes complications in simple language.",  # differently worded
    "How do I repair a motorcycle engine?",  # unsupported query
    "blue elephant window question",  # low-confidence / unclear query
    "My family member is unconscious and having a seizure."  # emergency
]

for i, q in enumerate(test_queries, 1):
    print(f"TEST {i}: {q}")
    print(diabetes_assistant(q))
    print("-" * 100)

## 11. Interactive assistant – ask multiple questions

Run this cell in Google Colab to start an interactive loop. Enter `quit` to stop. You can submit as many questions as you want during the session.

In [ ]:
print("DIABETES CARE INFORMATION ASSISTANT")
print("Ask a diabetes-related question. Type 'quit' to exit.\n")

while True:
    user_query = input("You: ").strip()
    if user_query.lower() in {"quit", "exit", "stop"}:
        print("Assistant: Thank you. Take care!")
        break
    print("\nAssistant:", diabetes_assistant(user_query), "\n")

## 12. Conclusion

This notebook implements an end-to-end cognitive assistant for diabetes information. It includes a labelled corpus, hierarchical taxonomy, domain concepts and relationships, NLP preprocessing, TF-IDF feature extraction, Logistic Regression classification, model evaluation, a domain knowledge base, safety rules and an interactive query interface.

### Limitations
- The dataset is small and manually created for an academic demonstration.
- The classifier may misclassify unfamiliar or ambiguous queries.
- The knowledge base is general and not a complete medical reference.
- The assistant does not diagnose, prescribe, calculate insulin doses or replace clinical care.

### Suggested future improvements
- Expand and clinically review the training corpus.
- Add multilingual support and more robust intent detection.
- Connect to verified, regularly updated public-health sources.
- Add clinician-reviewed safety testing and a user-friendly Gradio interface.

## References

1. World Health Organization (WHO). *Diabetes* — https://www.who.int/health-topics/diabetes
2. Centers for Disease Control and Prevention (CDC). *Diabetes* — https://www.cdc.gov/diabetes/
3. National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK). *Diabetes Overview* — https://www.niddk.nih.gov/health-information/diabetes/overview
4. scikit-learn documentation. *TfidfVectorizer* — https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
5. scikit-learn documentation. *LogisticRegression* — https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
6. NLTK documentation — https://www.nltk.org/